In [1]:
import gdsfactory as gf


from mesalab_pdk import get_pdk, LAYER

pdk = get_pdk()
pdk.activate()

In [5]:
from litho_markers import  ekst_ebl_pam_marker_arr

ekst_ebl_pam_marker_arr(shape=(13,13), boundary_margin=125).show()

In [2]:
from directional_couplers import coupler_imgrev
from directional_couplers import dc_lut
from cross_sections import xs_ekn300_te_IMGREV, xs_heater_metal

Ldc = dc_lut.get_length_um(gap_nm = 1000, width_um=1.5, ratio=0.9)
local_xs = gf.get_cross_section(xs_ekn300_te_IMGREV, width = 0.5)

dc = gf.partial(coupler_imgrev, gap = 1.0, length=Ldc, dy = 127, dx = 500, cross_section=local_xs, layer_core="WG", layer_trench="SIN_ETCH")
ekn_bend=gf.partial(gf.c.bend_euler, cross_section=local_xs, radius=300)

# mm = gf.components.mzi(splitter=dc, combiner=dc, cross_section=xs_ekn300_te_IMGREV,
#                        port_e0_splitter = "o2", port_e1_splitter = "o1",
#                        port_e0_combiner = "o2", port_e1_combiner = "o1",
#                        length_y = 10, length_x = 500, bend = ekn_bend).show()


# mm = gf.components.mzi2x2_2x2_phase_shifter(splitter=dc, combiner=dc, cross_section=local_xs,
#                        port_e0_splitter = "o2", port_e1_splitter = "o1",
#                        port_e0_combiner = "o2", port_e1_combiner = "o1",
#                        length_y = 10, length_x = 500, bend = ekn_bend).show()

In [3]:
from directional_couplers import coupler_ring_imgrev, coupler_imgrev
from cross_sections import xs_ekn300_te_IMGREV

xs = xs_ekn300_te_IMGREV(
    width=0.75,
    width_trench=50,
    layer="WG",
    layer_trench="SIN_ETCH",
    radius_min=10, radius=10,
)
d = gf.Component()

e = coupler_imgrev(dx = 500, dy = 127, cross_section=xs_ekn300_te_IMGREV, gap=0.5, length= 50, centered=True)



d.add_ref(e)
d.show()

In [4]:
# import gdsfactory as gf
# from gdsfactory.typings import CrossSectionSpec, ComponentSpec


# @gf.cell
# def ring_from_fixed_length_coupler(
#     splitter: ComponentSpec = "mmi1x2",
#     combiner: ComponentSpec | None = None,
#     target_length: float = 2000,
#     cross_section: CrossSectionSpec = "strip",
#     cross_section_x_top: CrossSectionSpec | None = None,
#     bend: ComponentSpec = "bend_euler",
#     **coupler_kwargs,
# ):
#     c = gf.Component()

#     dc = c.add_ref(splitter(**coupler_kwargs))

#     b1 = c.add_ref(bend(cross_section=cross_section))
#     b2 = c.add_ref(bend(cross_section=cross_section))
#     b3 = c.add_ref(bend(cross_section=cross_section))
#     b4 = c.add_ref(bend(cross_section=cross_section))


#     bend_length = b1.cell.info["length"]

#     # You need to define this reliably in your coupler wrapper
#     coupler_length = dc.cell.info["path_length"] 

#     if combiner is not None:
#         dc2 = c.add_ref(combiner(**coupler_kwargs))
#         second_arm_length = dc2.cell.info["path_length"] 
#     else:
#         second_arm_length = dc.dxsize

#     vertical_length = (
#         target_length
#         - coupler_length
#         - 4 * bend_length
#         - second_arm_length
#     ) / 2

#     if vertical_length < 0:
#         raise ValueError(
#             f"Target ring length {target_length} is too short. "
#             f"Minimum required is {coupler_length + 4*bend_length + second_arm_length:.3f} um."
#         )

#     s1 = c << gf.components.straight(
#         length=vertical_length,
#         cross_section=cross_section,
#     )
#     s2 = c << gf.components.straight(
#         length=vertical_length,
#         cross_section=cross_section,
#     )

#     #here we can put other dc or heater or so...

#     # connect from upper coupler ports; adapt names to your DC
#     b1.connect("o1", dc.ports["o3"])
#     s1.connect("o1", b1.ports["o2"])
#     b2.connect("o1", s1.ports["o2"])

#     b4.connect("o2", dc.ports["o2"])
#     s2.connect("o1", b4.ports["o1"])
#     b3.connect("o2", s2.ports["o2"])


#     #here we can put the other splitter / combiner
#     if combiner is None:
#         st = c << gf.components.straight(
#             length=second_arm_length,
#             cross_section=cross_section,
#         )
#         st.connect("o1", b2.ports["o2"])
#     else:
#         dc2.connect("o2", b2.ports["o2"])
#     # b3.connect("o2", st.ports["o2"])

#     # c.add_port("o1", port=dc.ports["o1"])
#     # c.add_port("o2", port=dc.ports["o2"])

#     c.info["target_length"] = target_length
#     c.info["actual_length"] = target_length
#     c.info["vertical_straight_length"] = vertical_length
#     c.info["coupler_ring_path_length"] = coupler_length

#     return c

In [5]:
from resonators import ring_from_fixed_length_coupler

In [6]:
a = ring_from_fixed_length_coupler(splitter=dc,
                                   combiner=dc,
                                   bend=ekn_bend,
                                   cross_section=local_xs,
                                   target_length=5820,
                                   minimum_section_length=1800)

a.show()